In [24]:
import os
from pathlib import Path
import re
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


In [ ]:

# Enforce reproducibility across stochastic processes
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)


In [27]:
# Centralized configuration class
class Config:
    # Data parameters
    VOCAB_SIZE = 5000
    MAX_LEN = 100
    BATCH_SIZE = 64
    
    # Model parameters
    EMBEDDING_DIM = 64
    HIDDEN_DIM = 128
    OUTPUT_DIM = 2
    
    # Training parameters
    EPOCHS = 6
    LR = 0.001
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # File Paths
    BASE_PATH = Path.cwd() / "imdb"
    TRAIN_PATH = BASE_PATH / "Train.csv"
    VALID_PATH = BASE_PATH / "Valid.csv"
    test_path = BASE_PATH / "Test.csv"

print(f"Systematic pipeline configured. Executing calculations on: {Config.DEVICE}")

Systematic pipeline configured. Executing calculations on: cpu


In [29]:
# 1. Load data partitions from disk
print("Loading CSV partitions...")
df_train = pd.read_csv(Config.TRAIN_PATH)
df_valid = pd.read_csv(Config.VALID_PATH)
df_test = pd.read_csv(Config.test_path)
print("Successfully done loading all the datasets")

Loading CSV partitions...
Successfully done loading all the datasets


In [30]:
# 2. Text preprocessing utility
def clean_and_tokenize(text):
    text = str(text).lower()
    text = re.sub(r'<br\s*/?>', ' ', text)  # Strip out HTML line breaks common in IMDB data
    text = re.sub(r'[^a-z\s]', '', text)     # Remove punctuation and numbers
    return text.split()

print("Tokenizing textual sequences...")
train_tokens = [clean_and_tokenize(t) for t in df_train['text']]
valid_tokens = [clean_and_tokenize(t) for t in df_valid['text']]
test_tokens = [clean_and_tokenize(t) for t in df_test['text']]

# 3. Build Vocabulary (Strictly from the training set to prevent leakage)
print("Building vocabulary indices...")
all_train_words = [word for text in train_tokens for word in text]
word_counts = Counter(all_train_words)

# Keep the most frequent words up to VOCAB_SIZE minus 1 (reserving 0 for padding)
common_words = [word for word, _ in word_counts.most_common(Config.VOCAB_SIZE - 1)]
word_to_idx = {word: idx + 1 for idx, word in enumerate(common_words)}
word_to_idx["<PAD>"] = 0  # Explicit padding index


Tokenizing textual sequences...
Building vocabulary indices...


In [34]:

# 4. Numericalization and Padding Pipeline
def pipeline_transform(tokenized_data, mapping, max_len):
    transformed_matrix = []
    for tokens in tokenized_data:
        # Convert tokens to integers if they exist in vocabulary, else ignore (OOV handling)
        numericalized = [mapping[w] for w in tokens if w in mapping]
        
        # Apply padding or truncation to normalize vector shapes
        if len(numericalized) < max_len:
            numericalized += [mapping["<PAD>"]] * (max_len - len(numericalized))
        else:
            numericalized = numericalized[:max_len]
        transformed_matrix.append(numericalized)
    return np.array(transformed_matrix)

print("Vectorizing sets into matching sequence formats...")
X_train = pipeline_transform(train_tokens, word_to_idx, Config.MAX_LEN)
X_valid = pipeline_transform(valid_tokens, word_to_idx, Config.MAX_LEN)
X_test = pipeline_transform(test_tokens, word_to_idx, Config.MAX_LEN)

# Extract integer label arrays
y_train = df_train['label'].values
y_valid = df_valid['label'].values
y_test = df_test['label'].values

print(f"Data mapping finalized. Vocab Size: {len(word_to_idx)} | Train Matrix Shape: {X_train.shape}")

Vectorizing sets into matching sequence formats...
Data mapping finalized. Vocab Size: 5000 | Train Matrix Shape: (40000, 100)
